# Funktionen / Code erklärt

## With-Funktion

Was macht `with` grundsätzlich? 
- es öffnet etwas, bearbeitet es, und schliesst es automatisch wieder
- ohne `with` müsste ich jedes Mal manuell schliessen (um Rechner nicht zu überlasten)

Beispiel:

In [4]:
bsp_raster = "S2C_MSIL2A_20250626T092051_N0511_R093_T35TLL_20250626T131216.SAFE/GRANULE/L2A_T35TLL_A004210_20250626T092359/IMG_DATA/R20m/T35TLL_20250626T092051_B01_20m.jp2"

In [5]:
import rasterio

with rasterio.open(bsp_raster) as src:
    data_20m = src.read(1)
    transform_20m = src.transform
    crs_20m = src.crs

Was passier hier wirklich?

- `with rasterio.open(bsp_raster) as src:` heisst: **`src = rasterio.open(bsp_raster)`**
- `data_20m = src.read(1)` & darauffolgende Zeilen innerhalb des Blocks--> damit arbeite ich
- nach dem Block wird die Datei automatisch geschlossen

## Automatisierter Bezug von Satellitedaten über API und STAC

### Begriffe / Erklärung

- **`Copernicus Data Space Ecosystem` = da wo die Daten liegen** -->  offizieller ESA Zugang zu Sentinel-2
- **`Microsoft Planetary Computer STAC API`** = ein anderer Metadaten-Katalog für Geodaten
- **`API` = Zugang zur Plattform CDSE** --> Werkzeuge die STAC nutzen oder ersetzen, um Daten zu suchen, zu filtern, zu laden oder zu verarbeiten
- **`STAC` = Standard, wie Satellitendaten beschrieben werden**


Das **Copernicus Data Space Ecosystem (CDSE)** ist die zentrale Plattform der ESA, über die Sentinel-2 Satellitendaten bereitgestellt werden. Es stellt den offiziellen Zugang zu den Datenprodukten der Copernicus-Mission dar.


Die **API** (Application Programming Interface) bildet den technischen Zugang zur Plattform CDSE. Sie ermöglicht es, automatisiert auf den Datenkatalog zuzugreifen und gezielt Daten abzufragen. Über die API können räumlich, zeitlich und inhaltlich gefilterte Datensätze gesucht, ausgewählt und heruntergeladen werden.


Der **STAC** (SpatioTemporal Asset Catalog) ist ein internationaler Standard zur Beschreibung von Geodaten. Er dient als strukturierter Katalog, der Satellitendaten systematisch beschreibt und auffindbar macht. STAC selbst enthält keine Bilddaten, sondern nur Metadaten und Verweise auf die eigentlichen Datenprodukte.


**Zusammengefasst**: Das Copernicus Data Space Ecosystem ist die Datenplattform, die API ist der Zugang zur Plattform, und STAC ist der standardisierte Katalog, der beschreibt, welche Daten verfügbar sind und wie sie strukturiert sind.

### Wie hängt was zusammen

Für diese Projektarbeit wird das **Copernicus Data Space Ecosystem (CDSE)** als Datenquelle für Sentinel-2 Satellitendaten verwendet. Die Plattform wird gewählt, da sie den offiziellen, langfristig stabilen Zugang zu den Copernicus-Daten der ESA bietet und sowohl den STAC-Katalog als auch eine API-basierte Schnittstelle zur automatisierten Datenabfrage bereitstellt.

Über die **API** des CDSE werden Satellitendaten der Sentinel-2 Mission automatisiert aus dem **STAC-Katalog** abgefragt. Die API dient dabei als technischer Zugangspunkt, über den gezielt räumlich und zeitlich gefilterte Datensätze gesucht, ausgewählt und heruntergeladen werden können.

Der **STAC-Katalog** stellt hierfür einen standardisierten Index bereit, der die Auffindbarkeit und Strukturierung der Satellitenprodukte vereinheitlicht. Auf dieser Grundlage können geeignete Sentinel-2 Szenen effizient identifiziert und die zugehörigen Bilddaten (z. B. spektrale Bänder) lokal in Python heruntergeladen und weiterverarbeitet werden.

Im Anschluss erfolgt die Zeitreihenanalyse, beispielsweise zur Ableitung von Vegetationsindizes wie dem NDVI.

### Was kann über STAC gefiltert werden?

- **`WO`**
    - **Bounding Box** (Rechteckgebiet über Koordinaten) ODER
    - **exakte Formen** statt Rechteck (Polygon) --> „Gib mir alle Szenen, die dieses Gebiet schneiden“
- **`WANN`**
    - **Datumsbereich** --> z.B. 2023-01-01 bis 2024-01-01
- **`WIE GUT`**
    - **Wolkenbedeckung** kann hier gefiltert werden --> z.B. `eo:cloud_cover < 20%`
- **Produkt**
    - z. B. nur: Sentinel-2 L2A (atmosphärenkorrigiert), nicht Rohdaten (L1C)

### Workflow mit STAC und API

1. Ich nutze die API
2. Die API greift auf den STAC-Katalog zu
3. STAC liefert Informationen darüber, welche Sentinel-2 Szenen existieren
4. Ich wähle passende Szenen nach Raum, Zeit und Qualität aus
5. Die API lädt die entsprechenden Datenprodukte aus dem Copernicus Data Space Ecosystem
6. In Python werden die relevanten spektralen Bänder extrahiert und weiterverarbeitet
7. Anschliessend erfolgt die Zeitreihenanalyse (z. B. NDVI)

1. STAC API
2. Cloud Data (Sentinel-2)
3. Cloud Processing (clip, mask, NDVI)
4. Export:
   - NDVI stacks (COG/Zarr)
   - oder summaries
5. Local machine:
   - time series analysis
   - plots
   - ML models